# 06 — Bubble wall tracking: 1+1D vs 3+1D

For each available time snapshot in both BubbleMaster (1+1D) and sledgehamr (3+1D),
finds the outer bubble wall position by locating where φ crosses the φ(R_out) threshold
from the instanton profile.  Compares wall trajectories to diagnose any systematic offset.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config  = pt.Config()
sh_base = '/pscratch/sd/b/buschman/other_runs/'

# --- BubbleMaster runs (one per unique lambda / collision geometry) ----------
# s=0 is Minkowski t=0 (nucleation); s=t_coll is when outer walls first touch.
BM_RUNS = [
    dict(lambda_bar=0.84,  d=55.55),
    dict(lambda_bar=0.069, d=12.54),
]

# --- sledgehamr runs (one entry per bubble to track) -------------------------
# sh_x_center: bubble centre in the box.  z_local = x - sh_x_center so that
# the outer-wall radius is measured directly from the bubble centre.
# Multiple entries with the same lambda_bar are overlaid on the same panel.
SH_BUBBLES = [
    # lambda = 0.84
    dict(lambda_bar=0.84, run_id='N3_234_0_1',
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_L=200., sh_axis='x', sh_x_center=127.8, sh_lev=0),
    dict(lambda_bar=0.84, run_id='N3_234_1_2',
         sh_path=sh_base+'bubble_N3_234_1_2_',
         sh_L=200., sh_axis='x', sh_x_center=127.8, sh_lev=0),
    # lambda = 0.069
    dict(lambda_bar=0.069, run_id='N3_456_0_1',
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_L=50.,  sh_axis='x', sh_x_center=28.77, sh_lev=0),
]

LAMBDA_VALS = sorted({b['lambda_bar'] for b in BM_RUNS}, reverse=True)

# Compute per-lambda physics quantities
lambda_info = {}
for r in BM_RUNS:
    lb   = r['lambda_bar']
    d    = r['d']
    model   = pt.PhysicsModel(pt.Phi4Potential(lb), config)
    t_coll  = float(model.kinematics.collision_time(d))
    gamma   = float(model.kinematics.Gamma(t_coll))
    profile = model.instanton
    lambda_info[lb] = dict(
        d=d, t_coll=t_coll, gamma=gamma,
        phi_threshold=float(profile.interp(profile.rout_0)),
        rout_0=profile.rout_0,
        bm_tag=f'lb{lb}_g{gamma:.4g}',
        model=model,
    )
    print(f'lb={lb}  d={d}  t_coll={t_coll:.3f}  '
          f'rout_0={profile.rout_0:.3f}  phi_thr={lambda_info[lb]["phi_threshold"]:.4f}')

## 1. BubbleMaster wall tracking

In [ ]:
def find_outer_wall(z, phi, threshold):
    """Return outermost z where phi > threshold; NaN if none."""
    idx = np.where(phi > threshold)[0]
    return float(z[idx[-1]]) if len(idx) > 0 else np.nan


bm_tracks = {}
for lb, info in lambda_info.items():
    tag   = info['bm_tag']
    model = info['model']
    out_dir = model.results_dir / 'validation' / tag / 'out'
    fields_path = out_dir / 'fields.h5'
    if not fields_path.exists():
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')
        continue
    with h5py.File(fields_path, 'r') as f:
        s_bm   = f['s'][:]
        z_bm   = f['z'][:]
        phi_bm = f['phi'][:]
    thr    = info['phi_threshold']
    d_half = info['d'] / 2.
    # s=0 is nucleation (Minkowski t=0); subtract d/2 to measure from bubble centre
    walls = np.array([find_outer_wall(z_bm, phi_bm[i, :], thr) - d_half
                      for i in range(len(s_bm))])
    bm_tracks[lb] = dict(t=s_bm, r_wall=walls)   # t = s directly (no shift)
    valid = np.isfinite(walls)
    print(f'lb={lb}  {valid.sum()}/{len(s_bm)} BM snapshots with wall  '
          f's=[{s_bm[0]:.3f},{s_bm[-1]:.3f}]')

## 2. sledgehamr wall tracking

In [ ]:
sh_tracks = {}   # keyed by run_id
for b in SH_BUBBLES:
    lb     = b['lambda_bar']
    info   = lambda_info[lb]
    thr    = info['phi_threshold']
    output = sledgehamr.Output(b['sh_path'])
    t_snaps = output.GetTimesOfSlices()

    t_out = []
    r_out = []
    n_fail = 0

    for idx in range(len(t_snaps)):
        try:
            slices = output.GetSlice(idx, b['sh_axis'], b['sh_lev'], ['Phi'])
            t      = slices['t']
            phi_2d = slices['Phi']
            n_rows, n_cols = phi_2d.shape
            phi_1d = phi_2d[n_rows // 2, :]
            z_loc  = np.linspace(0, b['sh_L'], n_cols) - b['sh_x_center']
            mask   = z_loc >= 0
            r      = find_outer_wall(z_loc[mask], phi_1d[mask], thr)
            t_out.append(t)
            r_out.append(r)
        except Exception:
            n_fail += 1

    t_arr = np.array(t_out)
    r_arr = np.array(r_out)
    sh_tracks[b['run_id']] = dict(lambda_bar=lb, t=t_arr, r_wall=r_arr)
    valid = np.isfinite(r_arr)
    print(f'[{b["run_id"]}]  {len(t_out)} loaded, {n_fail} skipped, '
          f'{valid.sum()} with wall'
          + (f'  t=[{t_arr[0]:.2f},{t_arr[-1]:.2f}]' if len(t_arr) else ''))

## 3. Plot

In [ ]:
colors = {0.84: '#0072B2', 0.069: '#D55E00'}
markers = ['o', 's', '^', 'D']   # one per SH_BUBBLES entry per lambda

fig, axes = plt.subplots(1, len(LAMBDA_VALS),
                          figsize=(5.5 * len(LAMBDA_VALS), 4), squeeze=False)

for col, lb in enumerate(LAMBDA_VALS):
    ax    = axes[0, col]
    color = colors[lb]
    info  = lambda_info[lb]

    # --- 3D tracks for this lambda (possibly multiple runs) ---
    runs_this_lb = [(b['run_id'], i)
                    for i, b in enumerate(SH_BUBBLES) if b['lambda_bar'] == lb]
    for mk_idx, (run_id, _) in enumerate(runs_this_lb):
        tr = sh_tracks.get(run_id)
        if tr is None:
            continue
        mask = np.isfinite(tr['r_wall'])
        ax.plot(tr['t'][mask], tr['r_wall'][mask],
                color=color, lw=1., label=f'3+1D ({run_id})',
                marker=markers[mk_idx % len(markers)], markersize=3,
                linestyle='none' if mask.sum() < 5 else '-', zorder=3)

    # --- BM track ---
    bm = bm_tracks.get(lb)
    if bm is not None:
        mask = np.isfinite(bm['r_wall'])
        ax.plot(bm['t'][mask], bm['r_wall'][mask],
                color=color, lw=1.5, ls='--', label='1+1D', zorder=3)

    # --- Free-expansion reference ---
    rout   = info['rout_0']
    t_coll = info['t_coll']
    # r(t) = sqrt(rout^2 + t^2) with t measured from nucleation (t=0=s=0)
    t_ref = np.linspace(0, t_coll * 1.6, 400)
    r_ref = np.sqrt(rout**2 + t_ref**2)
    ax.plot(t_ref, r_ref, 'k:', lw=1,
            label=r'free: $\sqrt{r_\mathrm{out}^2+t^2}$', zorder=2)

    ax.axvline(t_coll, color='grey', lw=0.8, ls=':')
    ax.text(t_coll * 1.01, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else rout * 0.5,
            '$t_\\mathrm{coll}$', fontsize=7, va='bottom', color='grey')

    ax.set_xlabel('Minkowski time $t$ (= BM time $s$, from nucleation)')
    ax.set_ylabel('Outer wall radius from bubble centre')
    ax.set_title(rf'$\bar\lambda = {lb}$', fontsize=9)
    ax.legend(frameon=False, fontsize=7)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_tracking.pdf', bbox_inches='tight')
plt.show()